In [5]:
import pandas as pd
from statsmodels.stats.inter_rater import fleiss_kappa
import numpy as np
from sklearn.metrics import (cohen_kappa_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score, matthews_corrcoef,precision_recall_curve, roc_curve, auc)
import os
import matplotlib.pyplot as plt

# =========================
# GLOBAL PARAMETERS
# =========================
INPUT_FILE = "Results_all_models.csv"

def bootstrap_compare(
    y_true,
    pred_a,
    pred_b,
    metric_fn,
    n_bootstrap=1000,
    seed=42
):
    """
    Bootstrap statistical comparison between two evaluators.

    Returns:
        - mean difference
        - 95% confidence interval
        - p-value
    """

    np.random.seed(seed)

    y_true = np.asarray(y_true)
    pred_a = np.asarray(pred_a)
    pred_b = np.asarray(pred_b)

    n = len(y_true)

    diffs = []

    for _ in range(n_bootstrap):

        idx = np.random.choice(
            n,
            n,
            replace=True
        )

        metric_a = metric_fn(
            y_true[idx],
            pred_a[idx]
        )

        metric_b = metric_fn(
            y_true[idx],
            pred_b[idx]
        )

        diffs.append(metric_a - metric_b)


    diffs = np.asarray(diffs)

    # Two-sided bootstrap p-value
    p_value = min(
        1.0,
        2 * min(
            np.mean(diffs <= 0),
            np.mean(diffs >= 0)
        )
    )

    return {
        "mean_diff": np.mean(diffs),
        "ci_low": np.percentile(diffs, 2.5),
        "ci_high": np.percentile(diffs, 97.5),
        "p_value": p_value
    }



def clean_evaluator_name(evaluator):
    """
    Converts:
    prediction_all-MiniLM-L6-v2_judge
    ->
    all-MiniLM-L6-v2
    """

    name = evaluator

    if name.startswith("prediction_"):
        name = name.replace(
            "prediction_",
            ""
        )

    if name.endswith("_judge"):
        name = name.replace(
            "_judge",
            ""
        )

    return name



def display_embedding_name(name):
    """
    Short names for paper tables.
    """

    if "all-MiniLM-L6-v2" in name:
        return "MiniLM"

    elif "all-mpnet-base-v2" in name:
        return "MPNet"

    elif "all-roberta-large-v1" in name:
        return "RoBERTa"

    elif "ensemble" in name.lower():
        return "ENSEMBLE"

    else:
        return name


def fpr_score(y_true, y_pred):
    """
    False Positive Rate = FP / (FP + TN)
    """
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    if (fp + tn) == 0:
        return 0.0

    return fp / (fp + tn)


def build_embedding_significance_table(
    df,
    embedding_evaluators,
    ground_truth="human_ensemble_judge",
    n_bootstrap=1000
):

    results = []


    for i, eval_a in enumerate(embedding_evaluators):

        for j, eval_b in enumerate(embedding_evaluators):

            if i >= j:
                continue


            model_a = display_embedding_name(
                clean_evaluator_name(eval_a)
            )

            model_b = display_embedding_name(
                clean_evaluator_name(eval_b)
            )


            required_cols = [
                ground_truth,
                eval_a,
                eval_b
            ]


            df_valid = df[
                required_cols
            ].dropna()


            if df_valid.empty:
                continue


            y_true = df_valid[ground_truth].values

            pred_a = df_valid[eval_a].values

            pred_b = df_valid[eval_b].values


            # =========================
            # F1 comparison
            # =========================

            f1_result = bootstrap_compare(
                y_true,
                pred_a,
                pred_b,
                metric_fn=f1_score,
                n_bootstrap=n_bootstrap
            )


            # =========================
            # Precision comparison
            # =========================

            precision_result = bootstrap_compare(
                y_true,
                pred_a,
                pred_b,
                metric_fn=precision_score,
                n_bootstrap=n_bootstrap
            )


            # =========================
            # Recall comparison
            # =========================

            recall_result = bootstrap_compare(
                y_true,
                pred_a,
                pred_b,
                metric_fn=recall_score,
                n_bootstrap=n_bootstrap
            )


            # =========================
            # FPR comparison
            # =========================

            fpr_result = bootstrap_compare(
                y_true,
                pred_a,
                pred_b,
                metric_fn=fpr_score,
                n_bootstrap=n_bootstrap
            )


            results.append({

                "Evaluator A": model_a,
                "Evaluator B": model_b,


                # -------------------------
                # F1
                # -------------------------

                "ΔF1": round(
                    f1_result["mean_diff"], 4
                ),

                "F1 95% CI": (
                    f"[{f1_result['ci_low']:.4f}, "
                    f"{f1_result['ci_high']:.4f}]"
                ),

                "F1 p-value": round(
                    f1_result["p_value"], 4
                ),


                # -------------------------
                # Precision
                # -------------------------

                "ΔPrecision": round(
                    precision_result["mean_diff"], 4
                ),

                "Precision 95% CI": (
                    f"[{precision_result['ci_low']:.4f}, "
                    f"{precision_result['ci_high']:.4f}]"
                ),

                "Precision p-value": round(
                    precision_result["p_value"], 4
                ),


                # -------------------------
                # Recall
                # -------------------------

                "ΔRecall": round(
                    recall_result["mean_diff"], 4
                ),

                "Recall 95% CI": (
                    f"[{recall_result['ci_low']:.4f}, "
                    f"{recall_result['ci_high']:.4f}]"
                ),

                "Recall p-value": round(
                    recall_result["p_value"], 4
                ),


                # -------------------------
                # FPR
                # -------------------------

                "ΔFPR": round(
                    fpr_result["mean_diff"], 4
                ),

                "FPR 95% CI": (
                    f"[{fpr_result['ci_low']:.4f}, "
                    f"{fpr_result['ci_high']:.4f}]"
                ),

                "FPR p-value": round(
                    fpr_result["p_value"], 4
                )

            })


    return pd.DataFrame(results)

    
# =========================
# USAGE EXAMPLE
# =========================

#semantic_ensemble()
    
df = pd.read_csv(INPUT_FILE)

#df2 = df.copy()
#df = jailbreakbench(df2)

model_names = ["llama-2-7b-chat","qwen2-7b-instruct","gpt-4o-mini","gemini-2.0-flash"]
tenses = ["present","past"]
turn_depths = [1,2,3]
human_judges_list = ["human_1_judge", "human_2_judge", "human_3_judge"]

subtopics = ["Hacking","Malware","Phishing"]

sources = ["AdvBench","HarmBench","Original"]

attacks = ["crescendo","tempest","mirage"]

# Example: only keep ASR and F1 metrics for detailed per-judge metrics. Full list on metrics_dict definition
#metrics_to_keep = ["F1 (weighted)","Cohen's Kappa","Recall","Precision","FPR","FNR","MCC","TN","TP","FN","FP"]
metrics_to_keep = ["F1 (weighted)","Recall","Precision","FPR","FNR","ROC AUC","PR AUC"]

# Filter only specific judges and metrics for detailed computation
"""selected_judges = ["prediction_all-MiniLM-L6-v2[no_chunk]_judge","prediction_all-MiniLM-L6-v2[chunked]_judge",
                   "prediction_all-mpnet-base-v2[no_chunk]_judge","prediction_all-mpnet-base-v2[chunked]_judge",
                   "prediction_all-roberta-large-v1[no_chunk]_judge","prediction_all-roberta-large-v1[chunked]_judge"
                  ]"""
selected_judges = [
                   "prediction_all-MiniLM-L6-v2[no_chunk]_judge",
                   "prediction_all-mpnet-base-v2[no_chunk]_judge",
                   "prediction_all-roberta-large-v1[no_chunk]_judge"
                  ]

selected_sources = ["AdvBench","HarmBench"]

model_name = model_names[0]
turn_depth = turn_depths[1]
tense = tenses[0]
subtopic = subtopics[1]
source = sources[1]
attack = attacks[0]

# Step 1: Filter + compute
"""df_metrics = compute_judge_metrics(
    df,
    model_filter=model_name,
    turn_depth_filter=turn_depth,
    tense_filter=tense,
    subtopic_filter=subtopic,
    source_filter=source,
    attack_type_filter=attack,
    judge_filter=selected_judges,
    metrics_filter=metrics_to_keep
)"""
"""df_metrics = compute_judge_metrics(
    df, judge_filter=selected_judges, source_filter=None, metrics_filter=metrics_to_keep)
#print(df_metrics)
# Step 2: Aggregate flexibly
df_agg = aggregate_metrics(
    df_metrics,
    #aggregate_by=["Judge","Turn Depth","Tense","Attack Type","Source"],
    aggregate_by=["Judge"]
)"""

# Save aggregated metrics
#df_agg.to_csv(f"detailed_result_agreement_metrics.csv", index=False)

#print("✅ Detailed and aggregated metrics saved.")

#plot_roc_pr_curves(df, selected_judges, ground_truth="human_ensemble_judge")

embedding_models = [
    "prediction_ensemble[chunked]_judge",
    "prediction_all-MiniLM-L6-v2[chunked]_judge",
    "prediction_all-mpnet-base-v2[chunked]_judge",
    "prediction_all-roberta-large-v1[chunked]_judge",
    #"prediction_ensemble[no_chunk]_judge",
    #"prediction_all-MiniLM-L6-v2[no_chunk]_judge",
    #"prediction_all-mpnet-base-v2[no_chunk]_judge",
    #"prediction_all-roberta-large-v1[no_chunk]_judge"
]


"""embedding_models = [
    "prediction_all-mpnet-base-v2[no_chunk]_judge",
    "prediction_all-mpnet-base-v2[chunked]_judge",
]
"""


"""embedding_models = [
    "prediction_ensemble[chunked]_judge",
    "prediction_all-mpnet-base-v2[no_chunk]_judge",
    "gpt-4.1_judge","gpt-5.2_judge",
    "rule-based_judge","llama-guard-3-8b_judge",
]"""


significance_results = build_embedding_significance_table(
    df,
    embedding_models,
    ground_truth="human_ensemble_judge",
    n_bootstrap=1000
)

# Save significance test results
SIGNIFICANCE_OUTPUT = "Embedding_significance_results.csv"
#SIGNIFICANCE_OUTPUT = "PromptG_compare__significance_results.csv"
#SIGNIFICANCE_OUTPUT = "Overall_significance_results.csv"

significance_results.to_csv(
    SIGNIFICANCE_OUTPUT,
    index=False
)

print(f"✅ Significance results saved to: {SIGNIFICANCE_OUTPUT}")

print(significance_results.to_string(index=False))


✅ Significance results saved to: Embedding_significance_results.csv
Evaluator A Evaluator B     ΔF1          F1 95% CI  F1 p-value  ΔPrecision   Precision 95% CI  Precision p-value  ΔRecall     Recall 95% CI  Recall p-value    ΔFPR         FPR 95% CI  FPR p-value
   ENSEMBLE      MiniLM  0.0065   [0.0038, 0.0091]       0.000      0.0067   [0.0041, 0.0093]                0.0   0.0042  [0.0003, 0.0081]           0.044 -0.0135 [-0.0190, -0.0077]          0.0
   ENSEMBLE       MPNet -0.0044 [-0.0074, -0.0015]       0.004     -0.0073 [-0.0103, -0.0043]                0.0   0.0050  [0.0010, 0.0093]           0.020  0.0198   [0.0133, 0.0266]          0.0
   ENSEMBLE     RoBERTa -0.0050 [-0.0090, -0.0009]       0.012     -0.0177 [-0.0214, -0.0142]                0.0   0.0308  [0.0248, 0.0373]           0.000  0.0571   [0.0497, 0.0647]          0.0
     MiniLM       MPNet -0.0109 [-0.0150, -0.0069]       0.000     -0.0140 [-0.0179, -0.0100]                0.0   0.0008 [-0.0052, 0.0065]         